# Sepsis DoR — mediation regression

Tests whether each org's effect on sensitivity is mediated by the
**test → discover → fix** funnel.

**Design.** Each run is a row. Outcome `sensitivity` (continuous). Three
run-level mediators: `T` (any agent tested end-to-end), `D` (any discovered
harm), `F` (any attempted fix). The validator enforces F=1 ⇒ D=1. One
pooled OLS with separate dummies for each org and each model; SEs
clustered at (model, org_name).

**Sequence.** For each org, observe the coefficient as funnel mediators are
progressively added:
- M0: `sens ~ C(org) + C(model)`
- M1: + T
- M2: + T + D
- M3: + T + D + F

**Per-org indirect effect** = M0 coef − M3 coef. Bootstrap CI on this
difference is reported per org.

## 1. Setup + data join

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

HERE = Path.cwd()
EXP_ROOT = HERE.parent.parent  # experiments/sepsis
RESULTS_ROOT = EXP_ROOT / "results"
INTERVENTIONS_CSV = EXP_ROOT / "analysis" / "interventions" / "all_results.csv"

rows = []
for p in sorted(RESULTS_ROOT.rglob("dor.json")):
    d = json.loads(p.read_text())
    anns = d["annotations"]
    rows.append({
        "model": d["model"],
        "org_name": d["org_name"],
        "run_id": d["run_id"],
        "T": int(max(a["tested_end_to_end"] for a in anns)),
        "D": int(max(a["discovered_harmful_outcomes"] for a in anns)),
        "F": int(max(a["attempted_fix"] for a in anns)),
    })
dor = pd.DataFrame(rows)

csv = pd.read_csv(INTERVENTIONS_CSV)
csv = csv[~csv["outcome"].isin(["timeout", "failure"])].dropna(subset=["sensitivity"])
df = dor.merge(
    csv[["model", "org_name", "run_id", "sensitivity"]],
    on=["model", "org_name", "run_id"], how="inner",
)
df["model_org"] = df["model"] + "/" + df["org_name"]

ORG_REF = "multi8_base"
OTHER_ORGS = ["multi8_comms", "multi8_dri", "multi8_safetydoc", "multi8_safetydocdri", "single_base"]
df = df[df["org_name"].isin([ORG_REF] + OTHER_ORGS)].copy()
df["org_name"] = pd.Categorical(df["org_name"], categories=[ORG_REF] + OTHER_ORGS)

print(f"runs: {len(df)} | models: {df['model'].nunique()} | orgs: {df['org_name'].nunique()} | clusters: {df['model_org'].nunique()}")
print(f"sens: {df['sensitivity'].mean():.3f} | T: {df['T'].mean():.3f} | D: {df['D'].mean():.3f} | F: {df['F'].mean():.3f}")
print("\ncell sizes (model x org):")
print(df.groupby(["model", "org_name"], observed=True).size().unstack(fill_value=0))

runs: 1054 | models: 5 | orgs: 6 | clusters: 30
sens: 0.778 | T: 0.691 | D: 0.555 | F: 0.455

cell sizes (model x org):
org_name                  multi8_base  multi8_comms  multi8_dri  \
model                                                             
claude-mythos-preview              30            28          27   
claude-opus-4-1-20250805           59            44          37   
claude-opus-4-6                    52            26          26   
claude-opus-4-7                    40            30          35   
claude-sonnet-4-6                  43            33          33   

org_name                  multi8_safetydoc  multi8_safetydocdri  single_base  
model                                                                         
claude-mythos-preview                   23                   27           31  
claude-opus-4-1-20250805                41                   38           49  
claude-opus-4-6                         40                   24           37  
claude-opus-4-7

## 2. Funnel descriptives per org

Per-org pass-through rates at each stage of the funnel: P(T), P(D|T), P(F|D).

In [2]:
def funnel(g):
    n = len(g)
    n_T = (g["T"] == 1).sum()
    n_D_given_T = ((g["T"] == 1) & (g["D"] == 1)).sum()
    n_D = (g["D"] == 1).sum()
    n_F_given_D = ((g["D"] == 1) & (g["F"] == 1)).sum()
    return pd.Series({
        "n": n,
        "P(T)": n_T / n if n else np.nan,
        "P(D|T)": n_D_given_T / n_T if n_T else np.nan,
        "P(F|D)": n_F_given_D / n_D if n_D else np.nan,
        "sens": g["sensitivity"].mean(),
    })

df.groupby("org_name", observed=True).apply(funnel, include_groups=False).round(3)

,n,P(T),P(D|T),P(F|D),sens
org_name,,,,,
multi8_base,224.0,0.438,0.571,0.298,0.609
multi8_comms,161.0,0.615,0.747,0.851,0.735
multi8_dri,158.0,0.981,0.910,0.908,0.871
multi8_safetydoc,191.0,0.440,0.548,0.413,0.762
multi8_safetydocdri,149.0,0.940,0.886,0.935,0.922
single_base,171.0,0.889,0.941,0.958,0.844


## 3. Waterfall — M0 → M3 per org

One pooled OLS at each spec. Each row of the waterfall is one org's
coefficient (vs `multi8_base` reference) as funnel mediators are
progressively added. Stars indicate p-values from cluster-robust SEs.

Funnel coefficients in M3 are reported in the second cell below.

In [3]:
SPECS = {
    "M0":           "sensitivity ~ C(org_name) + C(model)",
    "M1 (+T)":      "sensitivity ~ C(org_name) + T + C(model)",
    "M2 (+T+D)":    "sensitivity ~ C(org_name) + T + D + C(model)",
    "M3 (+T+D+F)":  "sensitivity ~ C(org_name) + T + D + F + C(model)",
}

def fit_clustered(formula, data):
    return smf.ols(formula, data=data).fit(
        cov_type="cluster", cov_kwds={"groups": data["model_org"]}
    )

fits = {label: fit_clustered(formula, df) for label, formula in SPECS.items()}

def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

waterfall = []
for org in OTHER_ORGS:
    key = f"C(org_name)[T.{org}]"
    row = {"org": org}
    for label, r in fits.items():
        coef = r.params[key]
        ci_lo, ci_hi = r.conf_int().loc[key]
        row[label] = f"{coef:+.3f}{stars(r.pvalues[key])} [{ci_lo:+.3f}, {ci_hi:+.3f}]"
    waterfall.append(row)
waterfall_df = pd.DataFrame(waterfall).set_index("org")
print("Org coefficients (vs multi8_base reference) at each spec; *** p<.001, ** p<.01, * p<.05\n")
print(waterfall_df.to_string())
print("\nR² by spec: " + "  ".join(f"{label}={r.rsquared:.3f}" for label, r in fits.items()))

Org coefficients (vs multi8_base reference) at each spec; *** p<.001, ** p<.01, * p<.05

                                             M0                     M1 (+T)                  M2 (+T+D)                M3 (+T+D+F)
org                                                                                                                              
multi8_comms            +0.125 [-0.018, +0.269]     +0.104 [-0.038, +0.246]    +0.099 [-0.041, +0.240]    +0.032 [-0.095, +0.159]
multi8_dri            +0.264** [+0.104, +0.425]    +0.197* [+0.038, +0.355]   +0.182* [+0.026, +0.338]    +0.064 [-0.054, +0.182]
multi8_safetydoc       +0.152* [+0.023, +0.282]    +0.153* [+0.030, +0.276]   +0.154* [+0.032, +0.277]  +0.144** [+0.037, +0.250]
multi8_safetydocdri  +0.318*** [+0.169, +0.467]  +0.255*** [+0.108, +0.402]  +0.242** [+0.096, +0.387]   +0.120* [+0.004, +0.236]
single_base           +0.241** [+0.090, +0.392]    +0.184* [+0.040, +0.327]   +0.168* [+0.022, +0.313]    +0.039 [-0.075, +0.154]



In [4]:
# Funnel coefficients in M3
m3 = fits["M3 (+T+D+F)"]
print("Funnel coefficients in M3:")
for v in ["T", "D", "F"]:
    coef = m3.params[v]
    lo, hi = m3.conf_int().loc[v]
    print(f"  {v}  {coef:+.4f}{stars(m3.pvalues[v])}  [{lo:+.3f}, {hi:+.3f}]  p={m3.pvalues[v]:.4f}")

Funnel coefficients in M3:
  T  +0.0962**  [+0.024, +0.168]  p=0.0092
  D  -0.1765***  [-0.228, -0.125]  p=0.0000
  F  +0.3578***  [+0.271, +0.445]  p=0.0000


## 4. Per-org bootstrap indirect-effect test

For each org, `indirect = M0 coef − M3 coef`. Cluster-bootstrap CIs and
two-sided p-values per org. Five per-org tests; Benjamini-Hochberg
correction controls the false discovery rate at 0.05.

In [5]:
rng = np.random.default_rng(0)
B = 1000
clusters = df["model_org"].unique()

boot_indirect = {o: np.empty(B) for o in OTHER_ORGS}
for b in range(B):
    drawn = rng.choice(clusters, size=len(clusters), replace=True)
    boot = pd.concat([df[df["model_org"] == c] for c in drawn], ignore_index=True)
    boot["org_name"] = pd.Categorical(boot["org_name"], categories=[ORG_REF] + OTHER_ORGS)
    try:
        r0 = smf.ols(SPECS["M0"], data=boot).fit()
        r3 = smf.ols(SPECS["M3 (+T+D+F)"], data=boot).fit()
        for o in OTHER_ORGS:
            key = f"C(org_name)[T.{o}]"
            boot_indirect[o][b] = r0.params.get(key, np.nan) - r3.params.get(key, np.nan)
    except Exception:
        for o in OTHER_ORGS:
            boot_indirect[o][b] = np.nan

# Per-org point estimates and bootstrap two-sided p-values
records = []
for o in OTHER_ORGS:
    key = f"C(org_name)[T.{o}]"
    m0_coef = fits["M0"].params[key]
    m3_coef = fits["M3 (+T+D+F)"].params[key]
    indirect = m0_coef - m3_coef
    arr = boot_indirect[o][~np.isnan(boot_indirect[o])]
    lo, hi = np.quantile(arr, [0.025, 0.975])
    # Two-sided bootstrap p-value: 2 * min(prop below 0, prop above 0)
    p_raw = 2 * min((arr <= 0).mean(), (arr >= 0).mean())
    pct = indirect / m0_coef * 100 if abs(m0_coef) > 0.01 else float("nan")
    records.append({"org": o, "m0": m0_coef, "m3": m3_coef, "indirect": indirect,
                    "lo": lo, "hi": hi, "p_raw": p_raw, "pct": pct})

# Benjamini-Hochberg adjustment across the 5 per-org tests
m_tests = len(records)
records_sorted = sorted(records, key=lambda r: r["p_raw"])
for k, r in enumerate(records_sorted, start=1):
    r["p_bh"] = min(r["p_raw"] * m_tests / k, 1.0)
# Enforce monotonicity (BH adjusted p must be monotone non-decreasing in raw p rank)
for k in range(len(records_sorted) - 2, -1, -1):
    records_sorted[k]["p_bh"] = min(records_sorted[k]["p_bh"], records_sorted[k+1]["p_bh"])

# Print in original org order
records_by_org = {r["org"]: r for r in records_sorted}
rows = []
for o in OTHER_ORGS:
    r = records_by_org[o]
    star = "*" if r["p_bh"] < 0.05 else ""
    rows.append({
        "org": o,
        "M0 coef": f"{r['m0']:+.3f}",
        "M3 coef": f"{r['m3']:+.3f}",
        "indirect": f"{r['indirect']:+.3f}",
        "95% CI": f"[{r['lo']:+.3f}, {r['hi']:+.3f}]",
        "p (raw)": f"{r['p_raw']:.3f}",
        "p (BH)": f"{r['p_bh']:.3f}{star}",
        "% mediated": f"{r['pct']:.0f}%" if not np.isnan(r['pct']) else "—",
    })
indirect_df = pd.DataFrame(rows).set_index("org")
print(f"Per-org indirect effect with cluster-bootstrap CIs.")
print(f"  p-values are two-sided bootstrap; BH correction across {m_tests} per-org tests.")
print(f"  * = BH-adjusted p < 0.05\n")
print(indirect_df.to_string())


Per-org indirect effect with cluster-bootstrap CIs.
  p-values are two-sided bootstrap; BH correction across 5 per-org tests.
  * = BH-adjusted p < 0.05

                    M0 coef M3 coef indirect            95% CI p (raw)  p (BH) % mediated
org                                                                                      
multi8_comms         +0.125  +0.032   +0.093  [+0.015, +0.190]   0.022  0.027*        74%
multi8_dri           +0.264  +0.064   +0.200  [+0.108, +0.308]   0.004  0.007*        76%
multi8_safetydoc     +0.152  +0.144   +0.009  [-0.060, +0.069]   0.688   0.688         6%
multi8_safetydocdri  +0.318  +0.120   +0.198  [+0.114, +0.304]   0.002  0.007*        62%
single_base          +0.241  +0.039   +0.202  [+0.108, +0.295]   0.004  0.007*        84%


## 5. Per-model M0 → M3

Same regression spec, stratified by model.

In [6]:
rows = []
for model in sorted(df["model"].unique()):
    sub = df[df["model"] == model].copy()
    sub["org_name"] = pd.Categorical(sub["org_name"], categories=[ORG_REF] + OTHER_ORGS)
    if sub["org_name"].nunique() < 2:
        continue
    try:
        r0 = smf.ols("sensitivity ~ C(org_name)", data=sub).fit()
        r3 = smf.ols("sensitivity ~ C(org_name) + T + D + F", data=sub).fit()
    except Exception:
        continue
    for org in OTHER_ORGS:
        key = f"C(org_name)[T.{org}]"
        if key not in r0.params:
            continue
        n_org = (sub["org_name"] == org).sum()
        c0, c3 = r0.params[key], r3.params[key]
        pct = (c0 - c3) / c0 * 100 if abs(c0) > 0.01 else float("nan")
        rows.append({
            "model": model.replace("claude-", ""),
            "org": org,
            "n": n_org,
            "M0": f"{c0:+.3f}",
            "M3": f"{c3:+.3f}",
            "% shrunk": f"{pct:.0f}%" if not np.isnan(pct) else "—",
        })
per_model = pd.DataFrame(rows)
print("Per-model M0 → M3 (each model's regression in isolation; no model FE since already stratified)\n")
print(per_model.to_string(index=False))

Per-model M0 → M3 (each model's regression in isolation; no model FE since already stratified)

            model                 org  n     M0     M3 % shrunk
   mythos-preview        multi8_comms 28 +0.542 +0.140      74%
   mythos-preview          multi8_dri 27 +0.789 +0.265      66%
   mythos-preview    multi8_safetydoc 23 +0.535 +0.572      -7%
   mythos-preview multi8_safetydocdri 27 +0.842 +0.317      62%
   mythos-preview         single_base 31 +0.595 +0.122      79%
opus-4-1-20250805        multi8_comms 44 +0.063 +0.071     -13%
opus-4-1-20250805          multi8_dri 37 -0.006 -0.050        —
opus-4-1-20250805    multi8_safetydoc 41 +0.105 +0.094      10%
opus-4-1-20250805 multi8_safetydocdri 38 +0.109 +0.061      44%
opus-4-1-20250805         single_base 49 +0.047 +0.008      84%
         opus-4-6        multi8_comms 26 -0.028 -0.104    -270%
         opus-4-6          multi8_dri 26 +0.376 -0.037     110%
         opus-4-6    multi8_safetydoc 40 +0.073 +0.072       1%
        

## 6. R² decomposition

Share of `sensitivity` variance captured at each stage of the regression
sequence: model identity alone, plus org dummies, plus the funnel mediators.

In [7]:
specs = {
    "model FE only":       "sensitivity ~ C(model)",
    "+ org dummies":       "sensitivity ~ C(model) + C(org_name)",
    "+ funnel (T+D+F)":    "sensitivity ~ C(model) + C(org_name) + T + D + F",
}
print(f"{'spec':>22s}  {'R²':>6s}  {'Δ R²':>6s}")
prev_r2 = None
for label, formula in specs.items():
    r = smf.ols(formula, data=df).fit()
    delta = f"{r.rsquared - prev_r2:+.3f}" if prev_r2 is not None else "—"
    print(f"  {label:>20s}  {r.rsquared:.3f}  {delta:>6s}")
    prev_r2 = r.rsquared


                  spec      R²    Δ R²
         model FE only  0.054       —
         + org dummies  0.210  +0.156
      + funnel (T+D+F)  0.359  +0.149


## 7. Notes

**Headline finding.** The safety-lead intervention raises sepsis sensitivity
by 0.26 points relative to the multi-agent baseline (M0 in §3, p < .01),
and approximately 76% of this effect is mediated by the
test → discover → fix funnel. Four of five non-baseline organizations
exhibit statistically significant funnel mediation under Benjamini-Hochberg
correction (§4); the safety-document intervention is the exception, with
no measurable mediation through the funnel. Within the funnel itself,
fixing dominates (F coefficient +0.36) and discovery without fixing is
strictly worse than no discovery at all (D coefficient −0.18, §3), implying
that observing harm without acting on it is a distinct failure mode rather
than a neutral intermediate state.

**Main body.** The waterfall in §3 (org coefficients across M0 → M3) and
the per-organization indirect-effect test in §4 jointly support the
headline mediation finding. The funnel-coefficient block in §3 carries
the within-funnel story.

**Appendix.** The per-model decomposition in §5 surfaces heterogeneity:
mythos-preview and Opus 4.7 carry the pooled signal because the other
three models have small or noise-dominated total effects. The R²
decomposition in §6 quantifies the additional variance attributable to
the funnel mediators (0.15 above model and organization fixed effects).

**Caveats to surface.** The total effect of organizational assignment is
identified by randomization. The mediated portion through the funnel
relies on sequential-ignorability assumptions that are not directly
testable; alternative mechanisms (framing effects from PM-level safety
language, role-induced caution in the lead's own coding, accountability
spillover via anticipated review) are consistent with the same observed
shrinkage. Standard errors are clustered at the (model, organization)
level (30 clusters); per-organization indirect-effect tests use
cluster-bootstrap confidence intervals to address the borderline-
cluster-count concern.